# **Phase 6 - Supervised Fine-Tuning Gemma 4**

In [ ]:
# %pip uninstall -y torch torchvision torchaudio unsloth unsloth_zoo bitsandbytes
# %pip cache purge

In [ ]:
# %pip install -q "torch==2.10.0"
# %pip install -q "torchvision==0.25.0"
# %pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
# %pip install -q "datasets>=3.4.1,<4.4.0"
# %pip install -q "trl>=0.18.2,<=0.24.0"
# %pip install -q peft bitsandbytes

In [2]:
import os
os.environ['PYTORCH_CUDA_ALLOC_CONF'] = 'expandable_segments:True'
os.environ['CUDA_VISIBLE_DEVICES']    = '0'

import warnings
warnings.filterwarnings('ignore', category=FutureWarning)

import torch
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.allow_tf32       = True

import gc
import json
import random
import threading
import numpy as np
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm
from concurrent.futures import ThreadPoolExecutor

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)

In [3]:
DATA_DIR = Path('workspace/data/cleaned')
SFT_DATA = Path('workspace/data/finetune/sft_data.jsonl')

MODEL_DIR = Path('workspace/models/gemma4')
MODEL_DIR.mkdir(parents=True, exist_ok=True)

## **1. Load & Sample Data**

In [ ]:
corpus      = pd.read_parquet(DATA_DIR / 'corpus.parquet')
train_split = pd.read_parquet(DATA_DIR / 'train_split.parquet')

corpus   = corpus.reset_index(drop=True)
cid2text = dict(zip(corpus['cid'].tolist(), corpus['text'].tolist()))

train_split['cid'] = train_split['cid'].apply(
    lambda x: [int(i) for i in (x.tolist() if isinstance(x, np.ndarray) else x)]
)
train_split['n_docs'] = train_split['cid'].apply(len)

In [ ]:
N_SAMPLES = 10_000

def build_context(cid_list: list) -> tuple[str, list[str]]:
    passages = [cid2text[c] for c in cid_list]
    parts    = []
    for i, p in enumerate(passages, 1):
        parts.append(f"[Văn bản {i}]\n{p}")
    return '\n\n'.join(parts), passages

sampled = (
    train_split
    .groupby('n_docs', group_keys=False)
    .apply(lambda g: g.sample(
        n=min(len(g), int(N_SAMPLES * len(g) / len(train_split)) + 1),
        random_state=SEED
    ))
    .sample(n=N_SAMPLES, random_state=SEED)
    .reset_index(drop=True)
)

sampled['ctx_str'], sampled['passages'] = zip(*sampled['cid'].apply(build_context))

sampled['prompt_len'] = sampled['ctx_str'].apply(lambda x: len(str(x)))
sampled               = sampled.sort_values('prompt_len').reset_index(drop=True)

## **2. Load Model**

In [ ]:
SYSTEM_PROMPT = """Bạn là một chuyên gia tư vấn pháp luật Việt Nam giàu kinh nghiệm. 
Nhiệm vụ của bạn là trả lời câu hỏi pháp lý dựa vào CHÍNH XÁC và DUY NHẤT vào các đoạn văn bản được cung cấp.

Quy tắc bắt buộc:
1. Chỉ sử dụng thông tin từ context đã cho — không dùng kiến thức ngoài.
2. Trả lời bằng tiếng Việt tự nhiên, rõ ràng, đúng ngữ pháp.
3. Cuối câu trả lời, liệt kê citations theo format:
   [Nguồn N: <trích dẫn ngắn gọn từ văn bản N>]
4. Nếu context không đủ để trả lời, nói rõ: "Thông tin trong văn bản chưa đủ để trả lời câu hỏi này."
5. KHÔNG bịa đặt thông tin, KHÔNG suy diễn ngoài văn bản.
6. KHÔNG xưng hô hay lòng vòng, đi thẳng vào câu trả lời.
7. KHÔNG sử dụng các cụm từ thể hiện bản thân là AI."""

In [ ]:
from transformers import AutoProcessor, AutoModelForCausalLM, BitsAndBytesConfig
import torch
import math

MODEL_PATH = 'unsloth/gemma-4-E4B-it'

bnb_config = BitsAndBytesConfig(
    load_in_4bit           = True,
    bnb_4bit_quant_type    = 'nf4',
    bnb_4bit_compute_dtype = torch.bfloat16,
)

processor = AutoProcessor.from_pretrained(MODEL_PATH)
processor.tokenizer.pad_token    = processor.tokenizer.eos_token
processor.tokenizer.padding_side = 'left'

model = AutoModelForCausalLM.from_pretrained(
    MODEL_PATH,
    quantization_config = bnb_config,
    device_map          = {'': 0},
    dtype               = torch.bfloat16,
    attn_implementation = 'sdpa',
).eval()

In [ ]:
def build_user_prompt(question: str, ctx_str: str) -> str:
    return f"""Câu hỏi: {question}\n\nVăn bản pháp luật liên quan:\n{ctx_str}"""

def generate_answer_batch_optimized(batch, model, device):
    questions = [row['question'] for row in batch]
    ctx_strs  = [row['ctx_str'] for row in batch]
    
    texts = []
    for q, c in zip(questions, ctx_strs):
        messages = [
            {'role': 'system', 'content': SYSTEM_PROMPT},
            {'role': 'user'  , 'content': build_user_prompt(q, c)},
        ]
        text = processor.apply_chat_template(
            messages, tokenize=False, add_generation_prompt=True, enable_thinking=False
        )
        texts.append(text)

    inputs = processor(
        text           = texts, 
        return_tensors = 'pt', 
        padding        = True, 
        truncation     = True,
        max_length     = 2048
    ).to(device)
    
    input_len = inputs['input_ids'].shape[-1]
    
    with torch.no_grad():
        outputs = model.generate(
            **inputs, 
            max_new_tokens = 256, 
            temperature    = 1.0, 
            top_p          = 0.95, 
            top_k          = 64,
            # use_cache      = True
        )
    
    generated_tokens = outputs[:, input_len:]
    return processor.batch_decode(generated_tokens, skip_special_tokens=True)

## **3. Generate Synthetic Dataset**

In [ ]:
BATCH_SIZE  = 64
sft_records = []

data_records = sampled.to_dict('records')
batches      = [data_records[i:i + BATCH_SIZE] for i in range(0, len(data_records), BATCH_SIZE)]

for batch in tqdm(batches, desc='Generating Synthetic Data'):
    answers = generate_answer_batch_optimized(batch, model, 'cuda:0')
    
    for row, answer in zip(batch, answers):
        sft_records.append({
            'qid': int(row['qid']),
            'cid': row['cid'],
            'messages': [
                {'role': 'system'   , 'content': SYSTEM_PROMPT},
                {'role': 'user'     , 'content': build_user_prompt(row['question'], row['ctx_str'])},
                {'role': 'assistant', 'content': answer},
            ],
        })

print(f"Total records: {len(sft_records)}")

In [ ]:
with open(SFT_DATA, 'w', encoding='utf-8') as f:
    for r in sft_records:
        f.write(json.dumps(r, ensure_ascii=False) + '\n')

In [ ]:
del model
gc.collect()
torch.cuda.empty_cache()

## **4. SFT + Unsloth + Gemma 4 E4B**

In [1]:
from unsloth      import FastLanguageModel

import torch
from datasets     import load_dataset
from trl          import SFTTrainer, SFTConfig
from transformers import TrainingArguments

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.
Unsloth: Your Flash Attention 2 installation seems to be broken. Using Xformers instead. No performance changes will be seen.
🦥 Unsloth Zoo will now patch everything to make training faster!


In [4]:
MODEL_PATH     = 'unsloth/gemma-4-E4B-it'
MAX_SEQ_LENGTH = 2048
DTYPE          = torch.bfloat16
LOAD_IN_4BIT   = True

In [15]:
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name     = MODEL_PATH,
    max_seq_length = MAX_SEQ_LENGTH,
    dtype          = DTYPE,
    load_in_4bit   = LOAD_IN_4BIT,
)

model = FastLanguageModel.get_peft_model(
    model,
    r                          = 16,
    target_modules             = ['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha                 = 16,
    lora_dropout               = 0,
    bias                       = 'none',
    use_gradient_checkpointing = 'unsloth',
    random_state               = 42,
    use_rslora                 = False,
    loftq_config               = None,
)

==((====))==  Unsloth 2026.4.7: Fast Gemma4 patching. Transformers: 5.5.0.
   \\   /|    NVIDIA RTX 6000 Ada Generation. Num GPUs = 1. Max memory: 47.372 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.10.0+cu128. CUDA: 8.9. CUDA Toolkit: 12.8. Triton: 3.6.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!


model.safetensors.index.json: 0.00B [00:00, ?B/s]

Fetching 3 files:   0%|          | 0/3 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/2130 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/208 [00:00<?, ?B/s]

processor_config.json: 0.00B [00:00, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/32.2M [00:00<?, ?B/s]

[unsloth_zoo.log|WARNING]Unsloth: Failed to register input-embedding hook for `model.base_model.model.model.audio_tower`: `get_input_embeddings` not auto‑handled for Gemma4AudioModel; please override in the subclass.. Falling back to pre-forward hook.


In [16]:
dataset = load_dataset('json', data_files={'train': str(SFT_DATA)})

def format_chat_template(example):
    text = tokenizer.apply_chat_template(
        example['messages'],
        tokenize              = False,
        add_generation_prompt = False
    )
    return {'text': text}

dataset = dataset.map(format_chat_template, batched=False)

Generating train split: 0 examples [00:00, ? examples/s]

Map:   0%|          | 0/3000 [00:00<?, ? examples/s]

In [17]:
trainer = SFTTrainer(
    model            = model,
    train_dataset    = dataset['train'],
    processing_class = tokenizer,

    args = SFTConfig(
        dataset_text_field = 'text',     
        max_length         = MAX_SEQ_LENGTH, 
        packing            = False,                 
        
        per_device_train_batch_size = 8,
        gradient_accumulation_steps = 2,
        warmup_steps                = 100,
        num_train_epochs            = 5,
        learning_rate               = 2e-4,
        fp16                        = False,
        bf16                        = True,
        logging_steps               = 50,
        optim                       = 'adamw_8bit',
        weight_decay                = 0.01,
        lr_scheduler_type           = 'linear',
        seed                        = 42,
        output_dir                  = str(MODEL_DIR),
    ),
)

trainer.train()

Unsloth: Tokenizing ["text"] (num_proc=64):   0%|          | 0/3000 [00:00<?, ? examples/s]

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'bos_token_id': 2}.
==((====))==  Unsloth - 2x faster free finetuning | Num GPUs used = 1
   \\   /|    Num examples = 3,000 | Num Epochs = 5 | Total steps = 940
O^O/ \_/ \    Batch size per device = 8 | Gradient accumulation steps = 2
\        /    Data Parallel GPUs = 1 | Total batch size (8 x 2 x 1) = 16
 "-____-"     Trainable parameters = 42,401,792 of 8,038,558,240 (0.53% trained)
Caching is incompatible with gradient checkpointing in Gemma4TextDecoderLayer. Setting `past_key_values=None`.


Unsloth: Will smartly offload gradients to save VRAM!


Step,Training Loss
50,7.853277
100,1.339894
150,0.839844
200,0.787534
250,0.726791
300,0.720681
350,0.717972
400,0.670707
450,0.644436
500,0.641231


Unsloth: Restored added_tokens_decoder metadata in workspace/models/gemma4/checkpoint-500/tokenizer_config.json.
Unsloth: Restored added_tokens_decoder metadata in workspace/models/gemma4/checkpoint-940/tokenizer_config.json.


TrainOutput(global_step=940, training_loss=1.059923070542356, metrics={'train_runtime': 6177.2362, 'train_samples_per_second': 2.428, 'train_steps_per_second': 0.152, 'total_flos': 4.3402725082709914e+17, 'train_loss': 1.059923070542356, 'epoch': 5.0})